In [ ]:
%%capture
%pip install pandas
%pip install torch
%pip install sentence-transformers
%pip install numpy
%pip install torch-geometric

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
from typing import Literal
from sentence_transformers import SentenceTransformer
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import SAGEConv

## Build the Graph Neural Network

First let's get our cleaned dataset so that we can initialize the nodes and edges of our graph.

In [ ]:
dataset = pd.read_csv("data/clean.csv")
dataset["Date"] = pd.to_datetime(dataset["Date"])

In [ ]:
dataset

### Build Interaction Edges

Each edge represents a directed message from one user to another. There are two ways an interaction is determined:
- Temporal interaction: pair of messages within a set time window between two **different** users
    - Lots of noise, but on average these are unique, true interactions
- Mention interaction: explicit mention to another user

**There may be a way to weight these interaction types becauase mentions are explicit and hence more confident as meaningful edges.**

First we need a way to convert discord usernames to IDs.

In [ ]:
user_df = dataset[["AuthorID", "Author"]].drop_duplicates().reset_index(drop=True)
username_to_id = user_df.set_index("Author")["AuthorID"].to_dict()
username_to_id

- Drop duplicate users (Authors) so we only have the unique users
- Reset the indices from `0-N` (`N` = number of users) and drop the old index column

Now we can find edges in our cleaned dataset.

Each edge will have the following information:
- edge_id
- source_user_id
- target_user_id
- timestamp
- edge_type (mention | temporal)
- metadata (time_delta, has_media)

In [ ]:
edges = []
T = pd.Timedelta(minutes=5)
last_index = None

In [ ]:
for i, row in dataset.iterrows():
    source_user = row["AuthorID"]
    timestamp = row["Date"]
    content = row["Content"]

    # mention edges
    if row["has_text"]:
        for username, target in username_to_id.items():
            if f"@{username}" in content and target != source_user:
                edges.append({
                    "source_user_id": source_user,
                    "target_user_id": target,
                    "timestamp": timestamp,
                    "edge_type": "mention",
                    #"metadata": {"has_media": bool} add later in v2 maybe
                })
    # temporal edges
    if last_index is not None:
        prev = dataset.loc[last_index]
        prev_source = prev["AuthorID"]
        prev_timestamp = prev["Date"]

        if source_user != prev_source and (timestamp - prev_timestamp) <= T:
            edges.append({
                "source_user_id": source_user,
                "target_user_id": prev_source,
                "timestamp": timestamp,
                "edge_type": "temporal",
                #"metadata": {"has_media": bool} add later in v2 maybe
            })
    
    last_index = i

In [ ]:
edges_df = pd.DataFrame(edges)
edges_df

### Build User Nodes

Each node will represent a user and we'll use `pandas` Dataframes to represent them.

First let's get the `avg_text_embedding` of each user.

In [ ]:
text_df = dataset[dataset["has_text"]].copy()
text_df

Clean the text content for links, this is **only** for embedding. We may need links in text for downstream tasks.

In [ ]:
URL_PATTERN = r"https?://\S+"

text_df["clean_text"] = (
    text_df["Content"]
    .str.replace(URL_PATTERN, "", regex=True)
    .str.strip()
)

Now we can use the `clean_text` column to create average text embeddings for each user node.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(
    text_df["clean_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
)

text_df["embeddings"] = list(embeddings)

In [ ]:
text_df.head()

Now we have an `embeddings` column that embeds the cleaned content from each message, let's take the average.

In [ ]:
avg_text_embedding = (
    text_df
    .groupby("AuthorID")["embeddings"]
    .apply(lambda x: np.mean(np.stack(x), axis=0))
)

This cell takes the `text_df`, groups it by `AuthorID` (user IDs) as indices that point to a `Series` of embeddings.

```
AuthorID → [emb₁, emb₂, emb₃, ...]
```

Then we use the `apply` method with a lambda function with parameter `x`, a `Series` of embeddings.

```
x = [array(d), array(d), array(d), ...]
x: Series[array(d), array(d), ...]
```

The lambda function stacks each array (embedding) from a user and gets the mean embedding by taking the mean of `axis=0`.
We use `stack` because NumPy can't get the mean array of a Series or arrays, hence we need to make a 2D array.

For example:
```python
x = Series[
  [0.1, 0.2, 0.3],
  [0.0, 0.4, 0.1],
  [0.2, 0.1, 0.5]
]

np.stack(x)

x = array([
  [0.1, 0.2, 0.3],
  [0.0, 0.4, 0.1],
  [0.2, 0.1, 0.5]
])

np.mean(np.stack(x), axis=0) # store the means of the first dimension (axis=0)

y = array[
    avg([0.1, 0.0, 0.2]),
    avg([0.2, 0.4, 0.1]),
    avg([0.3, 0.1, 0.5])
]
```

This gives us `y`, the average embedding for that user node.

In [ ]:
avg_text_embedding = (
    text_df
    .groupby("AuthorID")["embeddings"]
    .apply(lambda x: np.mean(np.stack(x), axis=0))
)
avg_text_embedding

Now we can create our `pandas` DataFrame that we'll use to store our user nodes.

In [ ]:
nodes_df = user_df.merge(
    avg_text_embedding,
    left_on="AuthorID", # join by matching AuthorID to...
    right_index=True, # match to the index of avg_text_embedding
    how="left" # keep all users in user_df, give NaN if no matching user in avg_textembedding
).rename(
    columns={
        "AuthorID": "user_id",
        "Author": "username",
        "embeddings": "avg_text_embedding",
    }
)
nodes_df

Now let's get the following activity stats from `edges_df`:
- interaction initiation count
- interaction reply count
- total message count

First we get the counts of outgoing and incoming edges for each user.

In [ ]:
source_df = edges_df[["source_user_id"]]
source_df.value_counts()

In [ ]:
target_df = edges_df[["target_user_id"]]
target_df.value_counts()

Now we get a Series of counts by getting the `value_counts` of the indexed column in each `source` and `target` DataFrame.

In [ ]:
source_counts = source_df["source_user_id"].value_counts()
target_counts = target_df["target_user_id"].value_counts()

The counts are in Series so they can be mapped to the new feature columns `initiation_count` and `target_count` in the `nodes_df`.

In [ ]:
nodes_df["initiation_count"] = (
    nodes_df["user_id"]
    .map(source_counts)
    .fillna(0)
    .astype(int)
)

nodes_df["target_count"] = (
    nodes_df["user_id"]
    .map(target_counts)
    .fillna(0)
    .astype(int)
)

See below for how the `map` method works.
- It uses the `user_id` from `nodes_df` as the index to the counts Series
- The new column in `nodes_df`is set to this mapping because it's in the same order as the `user_id` column in the Dataframe.

In [ ]:
nodes_df["user_id"].map(source_counts)

In [ ]:
nodes_df["user_id"].map(target_counts)

Finally we add the two interaction types together to get the `tota_interactions` for each user node. Save it to its own column in `nodes_df`.

In [ ]:
nodes_df["total_interactions"] = nodes_df["initiation_count"] + nodes_df["target_count"]
nodes_df

Before we build the graph itself, let's make sure out embeddings are all of shape (384,). The following Dataframe should be **empty**.

In [ ]:
lens = nodes_df["avg_text_embedding"].apply(len)
nodes_df[lens != 384]

## Build Node Index Representation

Let's map each node to a contiguous index that PyTorch Geometric can interpret.

In [ ]:
# get a stable, deterministic ordering of nodes
nodes_df = nodes_df.sort_values("user_id").reset_index(drop=True)

First we'll sort them so their order is the same everytime we run it with the same data. This is important for reproducibility.

In [ ]:
user_ids = nodes_df["user_id"].to_numpy()
user_id_to_index = pd.Series(
    index=user_ids,
    data=np.arange(len(user_ids))
).to_dict()

user_id_to_index

In [ ]:
mapping_df = nodes_df[["user_id", "username"]].copy()
mapping_df["node_index"] = np.arange(len(mapping_df))
mapping_df

## Build Edge Index Representation (Edge List)

Represent interactions (edges) as a `2 × E` edge list required by PyTorch Geometric.

First let's add information to the `edges_df` about the node indices each edge is connected to rather than the `user_id`.

In [ ]:
edges_df["source_node"] = edges_df["source_user_id"].map(user_id_to_index)
edges_df["target_node"] = edges_df["target_user_id"].map(user_id_to_index)

Let's do a quick check that there are no invalid node indices.

In [ ]:
assert edges_df["source_node"].notna().all()
assert edges_df["target_node"].notna().all()

In [ ]:
edges_df.head()

From this Dataframe, we want torch tensors (compatible with `PyG`) that represent edges. Let's create `edge_index`, a tensor where each row represents a **directed interaction** between two users:
```
source_user_id → target_user_id
```

However, Graph Neural Networks do not operate on user IDs or DataFrames.  
They operate on **dense integer-indexed tensors**.

In [ ]:
edge_index = torch.tensor(
    edges_df[["source_node", "target_node"]].to_numpy().T, # convert the 2 columned dataframe to a 2xn numpy array, then transpose it
    dtype=torch.long
)

The created `edge_index` is a tensor that encodes the **graph structure only**.

```
E = # of edges
edge_index ∈ ℕ^{2 × E}
```

In [ ]:
edge_index.shape

### Why we create it
- PyTorch Geometric requires this format for message passing
- It tells the model **which nodes can exchange information**
- It is the adjacency structure of the graph in tensor form

## Convert node features to a tensor

Now we need to extract node features into a matrix that the GNN can read from, we'll call this `x`.

For each node (user), concatenate:
- `avg_text_embedding` (384-D)
- Activity stats (small numeric features that we computed)

In [ ]:
emb = np.stack(nodes_df["avg_text_embedding"].values)
emb.shape

`emb` is our node feature matrix with **only** `avg_text_embedding`:
```
emb ∈ ℝ^{num_nodes × 384}
```

Now let's prepare our activity stats to be concatenated.

In [ ]:
stats = nodes_df[[
    "initiation_count",
    "target_count",
    "total_interactions"
]].values

Let's quickly normalize these stats so we get stable training and don't get crazy bias for large interaction volume nodes.

In [ ]:
eps = 1e-8

stats = np.log1p(stats)
stats = (stats - stats.mean(axis=0)) / (stats.std(axis=0) + eps)
stats.shape

The shape of this N-dim array should be
```
stats ∈ ℝ^{num_nodes × 3}
```

Now let's concatenate the two NumPy arrays.

In [ ]:
x_np = np.hstack([emb, stats])
x_np.shape

Should be
```
x ∈ ℝ^{num_nodes × 387}
```

In [ ]:
x = torch.tensor(x_np, dtype=torch.float)
x.shape

Finally we convert from a NumPy N-dim array to a PyTorch tensor (compatible with PyG).

**We now have:**
- `x`: node features
- `edge_index`: graph structure

Both are required to instantiate a **PyTorch Geometric `Data` object**, which represents a complete graph and is the format expect by GNN models.

In [ ]:
x.shape

In [ ]:
x

In [ ]:
edge_index.shape

In [ ]:
edge_index

## PyTorch Geometric

PyTorch Geometric (PyG) is a library for building and training Graph Neural Networks
using PyTorch.

It provides:
- Standard graph data containers (`Data`)
- Efficient message-passing operators
- Common GNN layers (GCN, GraphSAGE, GAT, etc.)

PyG expects graphs to be represented as **tensors**, not tables:
- `x`: node feature matrix
- `edge_index`: graph connectivity

With these, the entire graph can be processed efficiently on CPU or GPU using
neural message passing.


In [ ]:
# (optional) make the graph undirected
# if directionality is a signal, leave it directed
# edge_index = to_undirected(edge_index)

In [ ]:
data = Data(
    x=x,
    edge_index=edge_index
)

Now we are ready to create a GNN model that consume `Data` and learns node embeddings for each user.

Let's create a GNN that uses two GraphSAGE Convolution layers.

In [ ]:
class GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

This Graph Neural Network is a simple baseline model for our embedding task. It's an encoder that creates embedding representations of users with respect to other users. 
- The first convolutional layer maps the `in_dim` (387) of the input vector (`avg_text_embedding` concatenated with node activity stats) to the `hidden_dim` of the model
    - `hidden_dim` is a hyperparameter that can be thought of as an "expressiveness" level of the model (more hidden dims = more expressive model at risk of overfitting)
- Between layer we have a ReLU acitvation function so the graph can model non-linear functions (Graph propagation functions/transformations are linear by themselves)
- Finally the second convolutional layer maps the `hidden_dim` to a lower-dimensional output space (`out_dim`), producing the final user embedding that captures both behaviour and interaction of the node, but **not** message semantics
    - This embedding can be used for a multitude of tasks (classification, clustering, similarity)

### GraphSAGE

Let's get into how GraphSAGE works exactly. More specifically, how does the `SAGEConv` module perform neighborhood aggregation during message passing.

GraphSAGE updates each node’s embedding by **explicitly combining the node’s own features with information aggregated from its neighbors**.

If you know about Graph Convolutional Networks (GCNs), this differs from its formula:
$$
\mathbf{H}^{(k+1)} =
\sigma\!\left(
\tilde{\mathbf{D}}^{-\frac{1}{2}}
\tilde{\mathbf{A}}
\tilde{\mathbf{D}}^{-\frac{1}{2}}
\mathbf{H}^{(k)}
\mathbf{W}^{(k)}
\right)
$$

- GraphSAGE nodes keep an explicit representation of themselves and concatenate new information.
- GCNs pushes all learning into the initial embeddings which "smooths over" node embeddings as we propagate through the graph.

Here is the GraphSAGE formula:
$$
\mathbf{h}_v^{(k+1)} =
\sigma\!\left(
\mathbf{W}^{(k)}
\left[
\mathbf{h}_v^{(k)}
\;\Vert\;
\text{AGG}^{(k)}\!\left(\{\mathbf{h}_u^{(k)} : u \in \mathcal{N}(v)\}\right)
\right]
\right)
$$

- $\mathbf{h}_v^{(k+1)}$: embedding node $v$ at layer $k$
- $\mathcal{N}(v)$: set of neighbours of node $v$
- $\text{AGG}^{(k)}(*)$: aggregation function over neighbours
    - Common choices: mean, sum, max-pooling
- $[ \space * \space || \space * \space ]$: concatenation of self embedding (node $v$) and aggregated neighbour embeddings
    - **here is the explicit self and neighbour separation of GraphSAGE**
- $W^{(k)}$: learnable weight matrix at layer $k$, it learns how to combine self vs neighbour information
- $\sigma{(*)}$: sigma function that squeezes all values into [0,1] range, also a non-linear activation function

### How `SAGEConv` works

Each `SAGEConv` layer performs the following steps:

#### 1. Neighborhood aggregation
For a node (`u`), GraphSAGE collects the feature vectors of all neighboring nodes $ v \in \mathcal{N}(u) $ (node $v$ in all neighbours of $u$) as defined by `edge_index`.

These neighbor features are aggregated using a **permutation-invariant function** (a function in which input order doesn't affect output), which is mean aggregation by default in `SAGEConv`.

This produces a single vector representing the local neighborhood.

#### 2. Self-neighbour combination

The node's ($u$) **own feature vector** is kept separate and combined with the aggregate neighbour vector.
This design choice is critical because:
- It preserves node identity
- It prevents over-smoothing 
    - Repeated neighbour aggregation causes nodes to converge 
- It allows structurally different nodes to remain distinguishable
    - Over-smoothing causes nodes lose local structure and identity, they become indistinguishable from each other


$$
\mathbf{h}_u^{(k+1)} =
\sigma\!\left(
\mathbf{W}^{(k)}
\begin{bmatrix}
\mathbf{h}_u^{(k)} \\
\displaystyle \frac{1}{|\mathcal{N}(u)|}\sum_{v \in \mathcal{N}(u)} \mathbf{h}_v^{(k)}
\end{bmatrix}
\right)
$$

It's hard to see but the vector in brackets is a concatenation of $\mathbf{h}_u^{(k+1)}$ and all it's neighbours $\mathcal{N}(u)$.


#### 3. Linear transformation
The combined vector is then passed through a learned linear transformation (weight matrix and bias vector), projecting it into a new feature space (`hidden_dim`).


The formula for layer $k$ is:

$$
h_u^{(k)} = W^{(k)} \cdot \big[ h_u^{(k-1)} \;\|\; \text{AGG}(\{ h_v^{(k-1)} : v \in \mathcal{N}(u) \}) \big] + b^{(k)}
$$


### How this applies to our GNN class

- **First `SAGEConv` layer**
  - Inputs: raw node features (`avg_text_embedding` + activity stats)
  - Output: intermediate representations capturing **local interaction patterns**

- **ReLU activation**
  - Introduces non-linearity
  - Allows higher-order, non-linear interaction effects to be learned

- **Second `SAGEConv` layer**
  - Aggregates information from **neighbors-of-neighbors**
  - Produces the final user embeddings used for downstream tasks

Passing out `Data` object into the model, we get `z`, the tensor that represents a user embedding for each user node in our data.

In [ ]:
model = GNN(in_dim=data.x.shape[1], hidden_dim=128, out_dim=64)

z = model(data.x, data.edge_index)
z.shape

## Social Analysis

Now that we have our user embeddings we can draw conclusions from computed metrics.

Before we compute anything, let's first get a normalized version of `z`, `z_norm`. We don't want any magnitude bias in our computations.
- Normalize the L2
$$
normalize(x) = \frac{x}{||x||_{p}}; \space ||x||_{p} = (\sum_i{x_i}^{p})^{1/p}; \space p = 2
$$
- Normalize along `dim=1` because we wan't each node's feature vector normalized (dim 0, dim 1)

In [ ]:
z_norm = F.normalize(z, p=2, dim=1)
z_norm.shape, z.shape

### Centrality

Let's start with a simple one, centrality. This is a measure of how structurally important a user's role is within the group.

High centrality -> user interacts with many parts of the network
Low centrality -> peripheral role

```
centrality(u) = mean(cosine(z_u, z_all))
```

First we compute the cosine similarity matrix.

In [ ]:
sim = z_norm @ z_norm.T
sim.shape

Now `sim[i][j] = cosine(z_i, z_j)` because we already normalized by L2:

$$
z_i^{\text{norm}} = \frac{z_i}{||z_i||_2}
$$

Where

$$
||z_i||_2 = \sqrt{\sum_k{z_{i,k}^{2}}}
$$

So

$$
||z_i^{\text{norm}}||_2 = ||z_j^{\text{norm}}||_2 = 1 \\
$$

and `sim` is `z_norm` multipled by its own tranpose so

$$
\text{sim} = z_{\text{norm}} \, z_{\text{norm}}^\top \\
\text{sim}[i][j] = z_i^{\text{norm}} \cdot z_j^{\text{norm}}
$$

And thus, the similarity matrix value matches the cosine similarity formula

$$
\cos(z_i^{\text{norm}}, z_j^{\text{norm}})
= \frac{z_i^{\text{norm}} \cdot z_j^{\text{norm}}}{\|z_i^{\text{norm}}\|_2 \|z_j^{\text{norm}}\|_2}
= z_i^{\text{norm}} \cdot z_j^{\text{norm}}
$$

Now we can calculate the centrality of each user node represented in a vector.

In [ ]:
num_users = z.size(0)
centrality = (sim.sum(dim=1) - 1) / (num_users - 1)

centrality


Now we can add centrality to our `nodes_df` and sort to see which users are most central to the conversations.

In [ ]:
nodes_df["centrality"] = centrality.cpu().detach().numpy()
nodes_df.sort_values("centrality", ascending=False)


You should see the most acive people near the top and less active people near the bottom.

Chances are, most people are active in your group chat and the centrality of most people are very high and almost indistinguishable with negligible difference. Let's fine another metric that might show some variance through nuance.

### Bridge Score

**Definition**  
Bridge score measures how much a node connects *structurally different* neighbors.

**Computation**

Let $N(u)$ be the neighbors of node $u$, and $z_v$, the normalized embedding of neighbor $v$.

$$
\text{bridge}(u)
= 1 - \frac{1}{|N(u)|^2}
\sum_{v \in N(u)} \sum_{w \in N(u)} \cos(z_v, z_w)
$$

where

$$
\cos(z_v, z_w) = z_v^\top z_w
\quad \text{(embeddings are L2-normalized)}
$$

**Network Interpretation**

- $\text{bridge}(u) \approx 0$: neighbors are similar → user stays within their circle
- $\text{bridge}(u)$ high: neighbors are dissimilar → user connects groups


First let's get the neighbour messages of each node from our `edge_data`.

In [ ]:
# get edges
source, destination = data.edge_index

# get neighbours
neighbours = [[] for _ in range(data.num_nodes)] # data.num_nodes = x.shape[0]

for s, d in zip(source.tolist(), destination.tolist()):
    neighbours[s].append(d)

Now let's derive bridge scores for each node (user).

In [ ]:
scores = torch.zeros(data.num_nodes)

for u in range(data.num_nodes):
    ns = neighbours[u] # list of neighbour indices for each edge this node is connected to
    # bridge score needs at least two neighbours for bridge score
    if len(ns) < 2:
        scores[u] = 0.0
        continue
    
    Z = z_norm[ns]  # get list of normalized embeddings of the neighbour at each edge, shape: (edges, hidden_dim)
    sim = Z @ Z.T  # compute similarity matrix (cosine) amongst neighbors, shape: (edges, edges)
    scores[u] = 1.0 - sim.mean()  # bridge score = 1 - cohesion, where cohesion is the average neighbour-neighbour similarity

scores

Now we can take our bridge scores, and map them into our `nodes_df` and see the bridge score of each user sorted by descending bridge score.

In [ ]:
nodes_df["bridge_score"] = scores.cpu().detach().numpy()
nodes_df.sort_values("bridge_score", ascending=False)

You should see more variance in bridge scores than centrality scores. Draw conclusions as you see them in the data.